In [1]:
import numpy as np
import subprocess
import os
import pandas as pd
import matplotlib.pyplot as plt


# from ICESEE.scripts.plotting.scaling_plots_csv_details import analyze_icesee
# from ICESEE.scripts.plotting.scaling_plots import plot_metrics,extract_metrics_from_log
from ICESEE.scripts.plotting.scaling_line_bar_plots import analyze_icesee

In [2]:
def generate_latex_table(df, table_name, flag, output_file=None):
    """
    Generate a LaTeX table from a DataFrame for weak or strong scaling performance metrics.

    Parameters:
    df (pandas.DataFrame): DataFrame containing performance metrics with columns:
        - 'nodes': Number of nodes (integer)
        - 'ICESEE_ranks': Number of ICESEE ranks (integer)
        - 'ISSM_ranks': Number of ISSM ranks (integer)
        - 'Nens': Number of ensemble members (integer)
        - 'efficiency': Efficiency value (float)
        - 'speedup': Speedup value (float, required for strong scaling)
    table_name (str): Name or title for the table caption (LaTeX special characters should be escaped if necessary).
    flag (str): Type of scaling, either 'weak_scaling' or 'strong_scaling'.
    output_file (str, optional): Name of the output LaTeX file. If None, defaults to 'Weak_Scaling_Table.tex' for weak scaling
        or 'Strong_Scaling_Table.tex' for strong scaling.

    Returns:
    str: LaTeX code for the generated table.

    Raises:
    ValueError: If required columns are missing, DataFrame is empty, or flag is invalid.
    TypeError: If required columns have incorrect data types.
    IOError: If the output file cannot be written.
    """
    import pandas as pd

    # Validate flag
    valid_flags = ['weak_scaling', 'strong_scaling']
    if flag not in valid_flags:
        raise ValueError(f"Flag must be one of: {', '.join(valid_flags)}")

    # Set default output file based on flag
    if output_file is None:
        output_file = f"{flag.capitalize()}_Table.tex"

    # Define required columns based on flag
    required_columns = ['nodes', 'ICESEE_ranks', 'ISSM_ranks', 'Nens', 'efficiency']
    if flag == 'strong_scaling':
        required_columns.append('speedup')

    # Check if DataFrame is empty
    if df.empty:
        raise ValueError("DataFrame is empty")

    # Check for required columns
    if not all(col in df.columns for col in required_columns):
        raise ValueError(f"DataFrame must contain all required columns: {', '.join(required_columns)}")

    # Validate data types
    for col in ['nodes', 'ICESEE_ranks', 'ISSM_ranks', 'Nens']:
        if not pd.api.types.is_integer_dtype(df[col]):
            raise TypeError(f"Column '{col}' must contain integer values")
    for col in ['efficiency'] + (['speedup'] if flag == 'strong_scaling' else []):
        if not pd.api.types.is_float_dtype(df[col]):
            raise TypeError(f"Column '{col}' must contain float values")

    # Escape LaTeX special characters in table_name
    def escape_latex(text):
        """Escape LaTeX special characters in a string."""
        latex_special_chars = {
            '&': r'\&', '%': r'\%', '$': r'\$', '#': r'\#', '_': r'\_',
            '{': r'\{', '}': r'\}', '~': r'\textasciitilde{}', '^': r'\textasciicircum{}',
            '\\': r'\textbackslash{}'
        }
        return ''.join(latex_special_chars.get(c, c) for c in str(text))

    table_name_escaped = escape_latex(table_name)

    # Start LaTeX document
    latex_content = r"""\documentclass{article}
\usepackage{booktabs}
\usepackage{amsmath}
\usepackage{siunitx}
\begin{document}

\begin{table}[ht]
\centering
\caption{""" + (r"Weak Scaling " if flag == "weak_scaling" else r"Strong Scaling ") + table_name_escaped + r"""}
\begin{tabular}{cccc S[table-format=2.2]""" + (r" S[table-format=2.2]" if flag == "strong_scaling" else "") + r"""}
\toprule
{Nodes} & {ICESEE Ranks} & {ISSM Ranks} & {$N_{\text{ens}}$} & {Efficiency}""" + (r" & {Speedup}" if flag == "strong_scaling" else "") + r""" \\
\midrule
"""

    # Add table rows
    for _, row in df[required_columns].iterrows():
        nodes = row['nodes']
        icesee_ranks = row['ICESEE_ranks']
        issm_ranks = row['ISSM_ranks']
        nens = row['Nens']
        efficiency = f"{row['efficiency']:.2f}"
        if flag == "weak_scaling":
            latex_content += f"{nodes} & {icesee_ranks} & {issm_ranks} & {nens} & {efficiency} \\\\\n"
        else:
            speedup = f"{row['speedup']:.2f}"
            latex_content += f"{nodes} & {icesee_ranks} & {issm_ranks} & {nens} & {efficiency} & {speedup} \\\\\n"

    # Close LaTeX table and document
    latex_content += r"""\bottomrule
\end{tabular}
\end{table}

\end{document}
"""

    # Write to file with error handling
    try:
        with open(output_file, 'w') as f:
            f.write(latex_content)
    except IOError as e:
        raise IOError(f"Failed to write LaTeX file '{output_file}': {str(e)}")

    return latex_content

def form_new_columns(df,model_np,node_size,baseline_nens,flag):
    ICESEE_ranks = df['ranks'].values*(model_np+1)
    ISSM_ranks = model_np
    df['ICESEE_ranks'] = ICESEE_ranks
    df['ISSM_ranks'] = ISSM_ranks
    df['nodes'] = [1 if rank < node_size else int(np.ceil(rank / node_size)) for rank in ICESEE_ranks]
    if flag=="strong_scaling":
        nens_strong = baseline_nens*np.ones_like(ICESEE_ranks)
        df['Nens'] = nens_strong
        return df.copy()[['nodes', 'ICESEE_ranks', 'ISSM_ranks', 'Nens','speedup','efficiency']]
    elif flag=="weak_scaling":
        nens_weak = baseline_nens*df['ranks'].values
        df['Nens'] = nens_weak
        return  df.copy()[['nodes', 'ICESEE_ranks', 'ISSM_ranks', 'Nens','efficiency']]

In [3]:
model_np=4
analyze_icesee("strong_scaling_o.log", scaling_type="strong_scaling", model_np=model_np)

/Users/bkyanjo3/da_project/ICESEE/scripts/plotting/scaling_line_bar_plots.py:227: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels([f"{N:.0f}" for N in ax.get_yticks()], fontsize=18, fontweight="bold")
/Users/bkyanjo3/da_project/ICESEE/scripts/plotting/scaling_line_bar_plots.py:234: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels(yticklabels, fontsize=18, fontweight="bold")


{'outputs_dir': '/Users/bkyanjo3/da_project/ICESEE/applications/issm_model/examples/ISMIP_Choi/icesee_perf_outputs',
 'csvs': ['icesee_perf_outputs/strong_scaling_wallclock_scaling.csv',
  'icesee_perf_outputs/strong_scaling_forecast_scaling.csv',
  'icesee_perf_outputs/strong_scaling_raw_times.csv'],
 'plots': ['icesee_perf_outputs/strong_scaling_analysis_efficiency.png',
  'icesee_perf_outputs/strong_scaling_analysis_speedup.png',
  'icesee_perf_outputs/strong_scaling_forecast_efficiency.png',
  'icesee_perf_outputs/strong_scaling_forecast_speedup.png',
  'icesee_perf_outputs/strong_scaling_io_efficiency.png',
  'icesee_perf_outputs/strong_scaling_io_speedup.png',
  'icesee_perf_outputs/strong_scaling_stacked_breakdown.png',
  'icesee_perf_outputs/strong_scaling_wallclock_efficiency.png',
  'icesee_perf_outputs/strong_scaling_wallclock_speedup.png',
  'icesee_perf_outputs/weak_scaling_analysis_efficiency.png',
  'icesee_perf_outputs/weak_scaling_analysis_speedup.png',
  'icesee_perf_

In [6]:
path = 'icesee_perf_outputs/strong_scaling_wallclock_scaling.csv'
df = pd.read_csv(path)
df = form_new_columns(df,model_np=4,node_size=24,baseline_nens=64,flag="strong_scaling")
generate_latex_table(df, table_name="Wallclock", flag="strong_scaling", output_file="Strong_Scaling_wallclock_Table.tex")
df

,nodes,ICESEE_ranks,ISSM_ranks,Nens,speedup,efficiency
0,1,5,4,64,1.000000,100.000000
1,1,10,4,64,1.961098,98.054916
2,1,20,4,64,3.418778,85.469461
3,2,40,4,64,6.439078,80.488469
4,4,80,4,64,9.666762,60.417263
5,7,160,4,64,15.436604,48.239388
6,14,320,4,64,20.683687,32.318261


In [7]:
path = 'icesee_perf_outputs/strong_scaling_forecast_scaling.csv'
df = pd.read_csv(path)
df = form_new_columns(df,model_np=4,node_size=24,baseline_nens=64,flag="strong_scaling")
generate_latex_table(df,table_name="Forecast", flag="strong_scaling",output_file="Strong_Scaling_Forecast_Table.tex")
df


,nodes,ICESEE_ranks,ISSM_ranks,Nens,speedup,efficiency
0,1,5,4,64,1.000000,100.000000
1,1,10,4,64,1.969001,98.450041
2,1,20,4,64,3.513981,87.849533
3,2,40,4,64,6.915398,86.442471
4,4,80,4,64,12.199777,76.248609
5,7,160,4,64,19.235034,60.109482
6,14,320,4,64,28.433103,44.426723


In [8]:
path = 'icesee_perf_outputs/strong_scaling_analysis_scaling.csv'
df = pd.read_csv(path)
df = form_new_columns(df,model_np=4,node_size=24,baseline_nens=64,flag="strong_scaling")
generate_latex_table(df, table_name="Analysis", flag="strong_scaling", output_file="Strong_Scaling_analysis_Table.tex")
df

,nodes,ICESEE_ranks,ISSM_ranks,Nens,speedup,efficiency
0,1,5,4,64,1.000000,100.000000
1,1,10,4,64,1.262563,63.128142
2,1,20,4,64,1.480250,37.006239
3,2,40,4,64,1.667012,20.837646
4,4,80,4,64,1.846083,11.538016
5,7,160,4,64,2.249728,7.030400
6,14,320,4,64,2.726722,4.260503


In [9]:
path = 'icesee_perf_outputs/strong_scaling_io_scaling.csv'
df = pd.read_csv(path)
df = form_new_columns(df,model_np=4,node_size=24,baseline_nens=64,flag="strong_scaling")
generate_latex_table(df, table_name="I/O", flag="strong_scaling", output_file="Strong_Scaling_io_Table.tex")
df

,nodes,ICESEE_ranks,ISSM_ranks,Nens,speedup,efficiency
0,1,5,4,64,1.000000,100.000000
1,1,10,4,64,1.462290,73.114518
2,1,20,4,64,1.890397,47.259931
3,2,40,4,64,1.952073,24.400914
4,4,80,4,64,2.030019,12.687618
5,7,160,4,64,2.085196,6.516237
6,14,320,4,64,1.873407,2.927198


In [10]:
analyze_icesee("weak_scaling_o.log", scaling_type="weak_scaling", model_np=model_np)

/Users/bkyanjo3/da_project/ICESEE/scripts/plotting/scaling_line_bar_plots.py:227: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels([f"{N:.0f}" for N in ax.get_yticks()], fontsize=18, fontweight="bold")
/Users/bkyanjo3/da_project/ICESEE/scripts/plotting/scaling_line_bar_plots.py:234: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels(yticklabels, fontsize=18, fontweight="bold")


{'outputs_dir': '/Users/bkyanjo3/da_project/ICESEE/applications/issm_model/examples/ISMIP_Choi/icesee_perf_outputs',
 'csvs': ['icesee_perf_outputs/weak_scaling_wallclock_scaling.csv',
  'icesee_perf_outputs/weak_scaling_forecast_scaling.csv',
  'icesee_perf_outputs/weak_scaling_raw_times.csv'],
 'plots': ['icesee_perf_outputs/strong_scaling_analysis_efficiency.png',
  'icesee_perf_outputs/strong_scaling_analysis_speedup.png',
  'icesee_perf_outputs/strong_scaling_forecast_efficiency.png',
  'icesee_perf_outputs/strong_scaling_forecast_speedup.png',
  'icesee_perf_outputs/strong_scaling_io_efficiency.png',
  'icesee_perf_outputs/strong_scaling_io_speedup.png',
  'icesee_perf_outputs/strong_scaling_stacked_breakdown.png',
  'icesee_perf_outputs/strong_scaling_wallclock_efficiency.png',
  'icesee_perf_outputs/strong_scaling_wallclock_speedup.png',
  'icesee_perf_outputs/weak_scaling_analysis_efficiency.png',
  'icesee_perf_outputs/weak_scaling_analysis_speedup.png',
  'icesee_perf_output

In [11]:
path = 'icesee_perf_outputs/weak_scaling_wallclock_scaling.csv'
df = pd.read_csv(path)
df = form_new_columns(df,model_np=4,node_size=24,baseline_nens=8,flag="weak_scaling")
generate_latex_table(df, table_name="Wallclock", flag="weak_scaling", output_file="Weak_Scaling_wallclock_Table.tex")
df

,nodes,ICESEE_ranks,ISSM_ranks,Nens,efficiency
0,1,5,4,8,100.000000
1,1,10,4,16,96.369659
2,1,20,4,32,94.353624
3,2,40,4,64,86.691303
4,4,80,4,128,79.835197
5,7,160,4,256,71.349999
6,14,320,4,512,59.289217


In [12]:
path = 'icesee_perf_outputs/weak_scaling_forecast_scaling.csv'
df = pd.read_csv(path)
df = form_new_columns(df,model_np=4,node_size=24,baseline_nens=8,flag="weak_scaling")

generate_latex_table(df, table_name="Forecast", flag="weak_scaling", output_file="Weak_Scaling_Forecast_Table.tex")
df


,nodes,ICESEE_ranks,ISSM_ranks,Nens,efficiency
0,1,5,4,8,100.000000
1,1,10,4,16,93.310946
2,1,20,4,32,91.320089
3,2,40,4,64,85.453397
4,4,80,4,128,80.556932
5,7,160,4,256,75.346740
6,14,320,4,512,68.185741


In [ ]:
path = 'icesee_perf_outputs/weak_scaling_analysis_scaling.csv'
df = pd.read_csv(path)
df = form_new_columns(df,model_np=4,node_size=24,baseline_nens=8,flag="weak_scaling")
generate_latex_table(df,table_name="Analysis", flag="weak_scaling", output_file="Weak_Scaling_analysis_Table.tex")
df

: 

In [ ]:
path = 'icesee_perf_outputs/weak_scaling_io_scaling.csv'
df = pd.read_csv(path)
df = form_new_columns(df,model_np=4,node_size=24,baseline_nens=8,flag="weak_scaling")
generate_latex_table(df, table_name="I/O", flag="weak_scaling", output_file="Weak_Scaling_io_Table.tex")
df

: 

In [ ]:
# generate pdf's for all formed latex tables
  # List of LaTeX files to process
tex_files = [
    "Weak_Scaling_Table.tex",
    "Weak_Scaling_wallclock_Table.tex",
    "Weak_Scaling_Forecast_Table.tex",
    "Weak_Scaling_analysis_Table.tex",
    "Weak_Scaling_io_Table.tex",
    "Strong_Scaling_Table.tex",
    "Strong_Scaling_wallclock_Table.tex",
    "Strong_Scaling_Forecast_Table.tex",
    "Strong_Scaling_analysis_Table.tex",
    "Strong_Scaling_io_Table.tex"
]
for tex_file in tex_files:
    if os.path.exists(tex_file):
        try:
            subprocess.run(['pdflatex', tex_file], check=True)
            print(f"Successfully compiled {tex_file} to PDF.")
        except subprocess.CalledProcessError as e:
            print(f"Error compiling {tex_file}: {e}")

: 

: 

: 